# 模型观测构造

本单元把一条复权主力分钟序列的收益、RV 和 BPV 贡献，转换为既定采样区间上的模型观测。分钟贡献来自 c01 的 `compute_minute_contributions` 或调用者提供的同口径表；采样位置来自 b03，季节曲线来自 c01。函数不重新采样、估计季节性或拟合模型。

## 分钟处理与区间口径

`prepare_minute_observations` 独立完成分钟去季节。RV、BPV 分别使用调用者提供的每日曲线；不传曲线即不去季节。收益通过 `return_seasonality=None/'rv'/'bpv'` 选择：不调整，或除以对应方差因子的平方根。曲线是交易日 × 北京时间分钟末端钟点矩阵，调用者显式选择一个窗口，多个窗口分别调用。

$$\widetilde r_i=r_i/\sqrt{s_i},\qquad \widetilde {RV}_i=RV_i/s_i^{RV},\qquad \widetilde {BPV}_i=BPV_i/s_i^{BPV}.$$

BPV 使用已计算的原分钟贡献，不由调整后的相邻收益重新计算。贡献按照普通相邻行定义，不在午休、隔夜、换月或训练／测试分界切断。

`aggregate_model_observations` 只消费上游位置。源分钟行 i 占坐标区间 [i,i+1]，位置坐标为 `source_minute_index + interpolation_fraction`。每个分钟贡献在该行内按覆盖比例计入：完整分钟权重 1，端点分钟使用相应覆盖比例。休市不增加行或权重。分钟收益也使用这个积分口径；整数末端时等于复权分钟 close 的 log 差分汇总，分钟内结果不等同于 b03 的 open/close 插值价格差分。

## 使用边界与失败

分钟输入必须与位置生成时的完整实际分钟轴保持相同行序；不能删去缺失贡献或重新编号。每天不同 t 的序列各自独立；同一 t 的训练、测试两侧连续。位置顺序保留，要求同一 t 内坐标递增。

每个 t 的首个训练点只作为区间起点，观测为缺失并标记 `no_previous_sample`。测试侧首点若无前点，使用上游 t 的实际分钟 close 为起点；t 为完整输入第一分钟的 open 时使用坐标 0。因此正式全区间调用可以直接消费 b03 的训练为空、全部为测试侧的位置结果。

正权重范围内存在不可用分钟时，该指标的整个区间保留缺失和原因；权重为零的接触不影响结果。缺失曲线、非正因子、非法贡献和数值失败均显式记录，不填值。配置或位置无法对应分钟轴时抛出明确错误。函数不修改输入。

真实试错与完整 DLC demo 分章，需显式执行。定义仅放在 `export` 单元格，同名 Python 代理只加载这些定义。

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

In [ ]:
def prepare_minute_observations(
    minute_contributions_df: pd.DataFrame, *,
    rv_curve: pd.DataFrame | None = None,
    bpv_curve: pd.DataFrame | None = None,
    return_seasonality: str | None = None,
) -> pd.DataFrame:
    """连接每日曲线，先调整分钟收益与贡献；三个指标分别保留可用状态。"""
    if return_seasonality not in (None, "rv", "bpv"):
        raise ValueError("return_seasonality 必须为 None、rv 或 bpv")
    if return_seasonality == "rv" and rv_curve is None:
        raise ValueError("收益选择 RV 去季节时必须提供 rv_curve")
    if return_seasonality == "bpv" and bpv_curve is None:
        raise ValueError("收益选择 BPV 去季节时必须提供 bpv_curve")
    minute_observations_df = minute_contributions_df[["trading_date", "bar_at"]].reset_index(drop=True).copy()
    if not minute_observations_df["bar_at"].is_monotonic_increasing:
        raise ValueError("分钟观测须保持完整实际分钟轴的时间正序")
    trading_dates = pd.to_datetime(minute_observations_df["trading_date"]).dt.normalize()
    minute_slots = pd.to_datetime(minute_observations_df["bar_at"]).dt.tz_convert("Asia/Shanghai").dt.time
    seasonal_factors = {}
    for measure, curve_df in (("rv", rv_curve), ("bpv", bpv_curve)):
        factors = np.ones(len(minute_observations_df), dtype=float)
        if curve_df is not None:
            curve_dates = pd.DatetimeIndex(curve_df.index).normalize()
            day_positions = curve_dates.get_indexer(trading_dates)
            slot_positions = curve_df.columns.get_indexer(minute_slots)
            covered = (day_positions >= 0) & (slot_positions >= 0)
            factors[:] = np.nan
            curve_values = curve_df.to_numpy(dtype=float, na_value=np.nan)
            factors[covered] = curve_values[day_positions[covered], slot_positions[covered]]
        seasonal_factors[measure] = factors
        minute_observations_df[f"{measure}_seasonal_factor"] = factors
    return_factors = (np.ones(len(minute_observations_df)) if return_seasonality is None
                      else seasonal_factors[return_seasonality])
    minute_observations_df["return_seasonal_factor"] = return_factors

    for measure, contribution_col, factors in (
        ("log_return", "log_return", return_factors),
        ("rv", "rv_contribution", seasonal_factors["rv"]),
        ("bpv", "bpv_contribution", seasonal_factors["bpv"]),
    ):
        values = minute_contributions_df[contribution_col].to_numpy(dtype=float, na_value=np.nan)
        statuses = np.full(len(values), "ok", dtype=object)
        reasons = np.full(len(values), "", dtype=object)
        unavailable = ~np.isfinite(values)
        statuses[unavailable] = "missing_contribution"
        reasons[unavailable] = "分钟贡献缺失或非有限"
        if measure != "log_return":
            negative = np.isfinite(values) & (values < 0)
            statuses[negative] = "invalid_contribution"
            reasons[negative] = "RV/BPV 分钟贡献为负"
        missing_factor = (statuses == "ok") & ~np.isfinite(factors)
        statuses[missing_factor] = "missing_seasonality"
        reasons[missing_factor] = "该交易日或分钟位置没有有效季节因子"
        nonpositive_factor = (statuses == "ok") & (factors <= 0)
        statuses[nonpositive_factor] = "invalid_seasonality"
        reasons[nonpositive_factor] = "季节因子必须严格为正，零因子不能用于除法"
        adjusted_values = np.full(len(values), np.nan)
        valid = statuses == "ok"
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            denominators = np.sqrt(factors[valid]) if measure == "log_return" else factors[valid]
            adjusted_values[valid] = values[valid] / denominators
        numerical_failure = valid & ~np.isfinite(adjusted_values)
        statuses[numerical_failure] = "numerical_failure"
        reasons[numerical_failure] = "分钟去季节产生非有限结果"
        adjusted_values[numerical_failure] = np.nan
        minute_observations_df[measure] = adjusted_values
        minute_observations_df[f"{measure}_status"] = statuses
        minute_observations_df[f"{measure}_reason"] = reasons
    return minute_observations_df

## 复用既定采样边界

分钟准备结果可以供不同采样轴和尺度复用。区间输出保留全部采样位置字段，增加 `interval_start_coordinate`、`interval_end_coordinate`、`log_return`、`rv`、`bpv` 以及各指标的 `*_status`、`*_reason`。每行表示到该采样点为止的区间实现值；下一步预测仍由模型层按此序列完成。

输入从采样所用的全分钟轴中选取每分钟主力行，保持相同行序；仅过滤同分钟的其他合约，不过滤缺价行。多个 t 可一次输入，组内序列不得先删除失败点。函数保留区间失败行，下游模型据所需指标的状态决定是否具备拟合和预测条件。

In [ ]:
def _integrate_minute_observation(values, statuses, reasons, starts, ends):
    """按实际行坐标积分；仅正覆盖的分钟参与缺失判定。"""
    values = np.asarray(values, dtype=float)
    unavailable = ~np.isfinite(values) | (statuses != "ok")
    clean_values = np.where(unavailable, 0.0, values)
    missing_prefix = np.r_[0, np.cumsum(unavailable)]
    totals = np.full(len(starts), np.nan)
    output_statuses = np.full(len(starts), "no_previous_sample", dtype=object)
    output_reasons = np.full(len(starts), "本序列首个训练采样点没有前一区间起点", dtype=object)
    usable_rows = np.flatnonzero(np.isfinite(starts))
    if not len(usable_rows):
        return totals, output_statuses, output_reasons
    left = np.floor(starts[usable_rows]).astype(np.int64)
    right = np.ceil(ends[usable_rows]).astype(np.int64) - 1
    same_minute = left == right
    amounts = np.empty(len(usable_rows))
    with np.errstate(over="ignore", invalid="ignore"):
        amounts[same_minute] = (ends[usable_rows[same_minute]] - starts[usable_rows[same_minute]]) * clean_values[left[same_minute]]
        multiple = ~same_minute
        # 每个区间独立求和，避免从全局累计值相减时丢失小量贡献。
        interior_sum = np.zeros(len(usable_rows))
        has_interior = right > left + 1
        if np.any(has_interior):
            boundaries = np.column_stack((left[has_interior] + 1, right[has_interior])).ravel()
            interior_sum[has_interior] = np.add.reduceat(clean_values, boundaries)[::2]
        amounts[multiple] = (
            (left[multiple] + 1 - starts[usable_rows[multiple]]) * clean_values[left[multiple]]
            + (ends[usable_rows[multiple]] - right[multiple]) * clean_values[right[multiple]]
            + interior_sum[multiple]
        )
    totals[usable_rows] = amounts
    output_statuses[usable_rows] = "ok"
    output_reasons[usable_rows] = ""
    missing_rows = usable_rows[(missing_prefix[right + 1] - missing_prefix[left]) > 0]
    invalid_positions = np.flatnonzero(unavailable)
    if len(missing_rows):
        first_bad = invalid_positions[np.searchsorted(invalid_positions, np.floor(starts[missing_rows]).astype(np.int64))]
        output_statuses[missing_rows] = statuses[first_bad]
        output_reasons[missing_rows] = [
            f"源分钟行 {position}: {reason or '分钟观测缺失或非有限'}"
            for position, reason in zip(first_bad, reasons[first_bad])
        ]
        # 防止外部改写后的非有限值带着 ok 状态进入下游。
        output_statuses[missing_rows[output_statuses[missing_rows] == "ok"]] = "missing_contribution"
        totals[missing_rows] = np.nan
    failed = (output_statuses == "ok") & ~np.isfinite(totals)
    totals[failed] = np.nan
    output_statuses[failed] = "numerical_failure"
    output_reasons[failed] = "区间汇总产生非有限结果"
    return totals, output_statuses, output_reasons


def aggregate_model_observations(
    minute_observations_df: pd.DataFrame, sample_position_df: pd.DataFrame,
) -> pd.DataFrame:
    """将已准备分钟观测积分到各 t 的既定连续区间，保留采样行及失败原因。"""
    observations_df = sample_position_df.reset_index(drop=True).copy()
    minute_count = len(minute_observations_df)
    source_indices = observations_df["source_minute_index"].to_numpy(dtype=float, na_value=np.nan)
    fractions = observations_df["interpolation_fraction"].to_numpy(dtype=float, na_value=np.nan)
    if (not np.isfinite(source_indices).all() or np.any(source_indices != np.floor(source_indices))
        or np.any(source_indices < 0) or np.any(source_indices >= minute_count)):
        raise ValueError("source_minute_index 必须对应所提供完整分钟轴中的实际行")
    if not np.isfinite(fractions).all() or np.any((fractions < 0) | (fractions > 1)):
        raise ValueError("interpolation_fraction 必须是 [0,1] 内的有限值")
    source_indices = source_indices.astype(np.int64)
    minute_times = pd.DatetimeIndex(pd.to_datetime(minute_observations_df["bar_at"]))
    source_times = pd.DatetimeIndex(pd.to_datetime(observations_df["bar_at"]))
    if not np.array_equal(minute_times.take(source_indices).asi8, source_times.asi8):
        raise ValueError("位置的 bar_at 与源行不对应；不能切片、删行或重新编号分钟轴")
    ends = source_indices + fractions
    starts = pd.Series(ends).groupby(observations_df["t"], sort=False).shift().to_numpy(copy=True)
    # 训练为空的正式全区间采样仍有明确锚点，首个测试区间从 t 开始。
    first_test_rows = np.flatnonzero(~np.isfinite(starts) & observations_df["sample_side"].eq("test").to_numpy(dtype=bool))
    if len(first_test_rows):
        anchors = pd.DatetimeIndex(pd.to_datetime(observations_df.iloc[first_test_rows]["t"]))
        anchor_indices = minute_times.get_indexer(anchors)
        at_first_open = anchors == minute_times[0] - pd.Timedelta(minutes=1)
        if np.any((anchor_indices < 0) & ~at_first_open):
            raise ValueError("首个测试区间的 t 必须为源分钟 close 或完整输入第一分钟的 open")
        starts[first_test_rows] = np.where(at_first_open, 0, anchor_indices + 1)
    valid_starts = np.isfinite(starts)
    if np.any(starts[valid_starts] >= ends[valid_starts]):
        raise ValueError("每个 t 内采样位置须递增，区间必须具有正覆盖长度")
    observations_df["interval_start_coordinate"] = starts
    observations_df["interval_end_coordinate"] = ends
    for measure in ("log_return", "rv", "bpv"):
        values = minute_observations_df[measure].to_numpy(dtype=float, na_value=np.nan)
        statuses = minute_observations_df[f"{measure}_status"].to_numpy()
        reasons = minute_observations_df[f"{measure}_reason"].to_numpy()
        totals, interval_statuses, interval_reasons = _integrate_minute_observation(values, statuses, reasons, starts, ends)
        observations_df[measure] = totals
        observations_df[f"{measure}_status"] = interval_statuses
        observations_df[f"{measure}_reason"] = interval_reasons
    return observations_df

## 显式选择模型输入尺度

`transform_model_input(observations_df, observation_col=..., transform='identity'/'log', zero_replacement=None, require_nonnegative=False)` 选择一个观测列，返回保留原列的副本，并增加 `model_input`、`model_input_status`、`model_input_reason`。

默认保留输入尺度和零值。调用者若明确指定正数 `zero_replacement`，仅替换原值恰为零的有效观测；未指定时 log(0) 返回 `invalid_transform`，负值也不能取 log。无效区间不会因为零值转换变成有效。自然 log 结果保持 log 尺度，不进行指数还原。`require_nonnegative=True` 用于 MEM 等非负输入场景，转换后为负则返回 `negative_model_input`；不会自动改变变换或模型。
`initial_model_input_position(model_input_df)` 返回首次 `model_input_status == 'ok'` 且输入有限的位置；全序列无有效观测时返回序列长度。逐点调用者在完整既定序列上定位一次，从该位置积累初始训练历史；目标行仍全部保留。开始积累之后的内部缺失原样进入训练窗口，不在每个窗口里重新寻找有效前缀，也不拼接缺失两侧观测。每日独立重采样的调用者仅在覆盖全局源分钟起始处时处理结构性起始贡献缺失，其他训练窗口保留缺失。


In [ ]:
def transform_model_input(
    observations_df: pd.DataFrame, *, observation_col: str,
    transform: str = "identity", zero_replacement: float | None = None,
    require_nonnegative: bool = False,
) -> pd.DataFrame:
    """选择观测并显式处理零值和 log；保留已有失败，不替换模型或填缺失。"""
    if observation_col not in ("log_return", "rv", "bpv"):
        raise ValueError("observation_col 必须为 log_return、rv 或 bpv")
    if transform not in ("identity", "log"):
        raise ValueError("transform 必须为 identity 或 log")
    if zero_replacement is not None and (not np.isfinite(zero_replacement) or zero_replacement <= 0):
        raise ValueError("zero_replacement 必须为调用者明确指定的有限正数")
    model_input_df = observations_df.copy()
    values = observations_df[observation_col].to_numpy(dtype=float, na_value=np.nan).copy()
    statuses = observations_df[f"{observation_col}_status"].to_numpy(copy=True)
    reasons = observations_df[f"{observation_col}_reason"].to_numpy(copy=True)
    unavailable = (statuses == "ok") & ~np.isfinite(values)
    statuses[unavailable] = "missing_observation"
    reasons[unavailable] = "采样观测缺失或非有限"
    valid = statuses == "ok"
    if zero_replacement is not None:
        values[valid & (values == 0)] = zero_replacement
    if transform == "log":
        nonpositive = valid & (values <= 0)
        statuses[nonpositive] = "invalid_transform"
        reasons[nonpositive] = "log 输入必须严格为正；零值转换由调用者显式指定"
        valid = statuses == "ok"
        values[valid] = np.log(values[valid])
    if require_nonnegative:
        negative = (statuses == "ok") & (values < 0)
        statuses[negative] = "negative_model_input"
        reasons[negative] = "所选模型要求非负输入，转换后的观测为负"
    values[statuses != "ok"] = np.nan
    model_input_df["model_input"] = values
    model_input_df["model_input_status"] = statuses
    model_input_df["model_input_reason"] = reasons
    return model_input_df

def initial_model_input_position(model_input_df: pd.DataFrame) -> int:
    """定位完整序列的初始有效起点；调用方在切训练窗口前只定位一次，内部缺失原样保留。"""
    values = model_input_df["model_input"].to_numpy(dtype=float, na_value=np.nan)
    valid = model_input_df["model_input_status"].eq("ok").to_numpy(dtype=bool) & np.isfinite(values)
    positions = np.flatnonzero(valid)
    return int(positions[0]) if len(positions) else len(model_input_df)


In [ ]:
from config.data_contracts import FUTURES_MINUTE_SCHEMA

OBSERVATION_COLUMN_LABELS = {
    field.name: f"{field.name}（{field.metadata[b'field_name_zh'].decode('utf-8')}）"
    for field in FUTURES_MINUTE_SCHEMA if field.metadata and b"field_name_zh" in field.metadata
}


OBSERVATION_COLUMN_LABELS.update({
    "sample_id": "sample_id（采样点编号）", "t": "t（采样分界时点）",
    "sample_side": "sample_side（训练或测试侧）", "sample_number": "sample_number（侧内采样序号）",
    "source_minute_index": "source_minute_index（源分钟行号）",
    "interpolation_fraction": "interpolation_fraction（分钟内比例）",
    "sample_at": "sample_at（采样时点）", "main_contract_code": "main_contract_code（当时主力合约）",
    "interval_start_coordinate": "interval_start_coordinate（区间起点行坐标）",
    "interval_end_coordinate": "interval_end_coordinate（区间终点行坐标）",
    "log_return": "log_return（收益观测）", "rv": "rv（RV 观测）", "bpv": "bpv（BPV 观测）",
    "return_seasonal_factor": "return_seasonal_factor（收益使用的方差季节因子）",
    "rv_seasonal_factor": "rv_seasonal_factor（RV 季节因子）",
    "bpv_seasonal_factor": "bpv_seasonal_factor（BPV 季节因子）",
    "model_input": "model_input（所选模型输入）",
    "model_input_status": "model_input_status（模型输入状态）",
    "model_input_reason": "model_input_reason（模型输入失败原因）",
})
for _measure, _label in (("log_return", "收益"), ("rv", "RV"), ("bpv", "BPV")):
    OBSERVATION_COLUMN_LABELS[f"{_measure}_status"] = f"{_measure}_status（{_label}观测状态）"
    OBSERVATION_COLUMN_LABELS[f"{_measure}_reason"] = f"{_measure}_reason（{_label}观测失败原因）"

In [ ]:
import json
import pyarrow as pa
import pyarrow.parquet as pq
from R04_Research.a01_Methods.b02_MainContinuousAdjustment.c01_MainContinuousAdjustment import arrow_content_sha256
from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import (
    _load_intraday_demo_inputs, _compute_intraday_demo)
from R04_Research.a01_Methods.b03_Sampling.c01_SamplingPositions import (
    SAMPLING_POSITION_SCHEMA, generate_sampling_column, calibrate_sampling_unit,
    prepare_sampling_axis, generate_sampling_positions)



def _load_demo_sampling_axis(root, *, start, end):
    """按完整交易月只读真实全合约复权产物，直接调用正式 GK×成交额定义。"""
    from pathlib import Path
    import pyarrow.dataset as ds
    root = Path(root)
    source = root / "R04_Research/a01_Methods/b02_MainContinuousAdjustment/c01_MainContinuousAdjustment_DemoData" / (
        "c01_MainContinuousAdjustment_RB_train20100101-20241231_test20250101-20260930_ratio1.10_log_common_demo.parquet")
    columns = ["exchange_code", "underlying_code", "trading_date", "bar_at", "main_contract_code", "is_main", "money",
               "adjusted_open", "adjusted_high", "adjusted_low", "adjusted_close"]
    dataset, parts = ds.dataset(source, format="parquet"), []
    first, last = pd.Timestamp(start), pd.Timestamp(end)
    for month in pd.period_range(first, last, freq="M"):
        left, right = max(first, month.start_time), min(last, month.end_time.normalize())
        selected = (ds.field("trading_date") >= left.date()) & (ds.field("trading_date") <= right.date())
        monthly = dataset.to_table(filter=selected, columns=columns).to_pandas()
        if len(monthly):
            parts.append(generate_sampling_column(monthly, axis="gk_money", quantity_scope="all_contracts"))
    if not parts:
        raise ValueError("所选真实区间没有采样增量")
    return pd.concat(parts, ignore_index=True).sort_values("bar_at").reset_index(drop=True)


def _compute_observations_demo(demo_contributions, demo_seasonality, *, test_start,
                               exchange_code="XSGE", configurations=None, sampling_axis=None, axis_name="gk_money", N=15):
    """真实完整分钟轴采样、去季节及区间积分检查；不写文件。"""
    demo_test_start, demo_variety = pd.Timestamp(test_start), "RB"
    demo_window_names = tuple(demo_seasonality["rv"]["curves"])
    demo_curves = {window: {measure: demo_seasonality[measure]["curves"][window]
                  for measure in ("rv", "bpv")} for window in demo_window_names}
    if sampling_axis is None:
        if axis_name != "time":
            raise ValueError("sigma×成交额必须提供由正式全合约 OHLC／money 生成的分钟轴")
        demo_axis = generate_sampling_column(demo_contributions.assign(
            exchange_code=exchange_code, underlying_code=demo_variety), axis="time", quantity_scope="all_contracts")
    else:
        demo_axis = sampling_axis
    if not pd.DatetimeIndex(demo_axis["bar_at"]).equals(pd.DatetimeIndex(demo_contributions["bar_at"])):
        raise ValueError("采样轴和分钟贡献必须覆盖同一个完整分钟序列")
    demo_calibration = calibrate_sampling_unit(demo_axis, N=N, test_start_date=demo_test_start.date())
    demo_prepared_axis = prepare_sampling_axis(demo_axis, a=demo_calibration["a"], sampling_method="log_interpolate")
    demo_anchor = demo_axis.loc[demo_axis["trading_date"] < demo_test_start.date(), "bar_at"].iloc[-1]
    demo_positions = generate_sampling_positions(demo_prepared_axis, t=demo_anchor, K=None)
    demo_observation_schema = pa.schema([*SAMPLING_POSITION_SCHEMA,
        *[pa.field(name, pa.float64()) for name in
          ("interval_start_coordinate", "interval_end_coordinate", "log_return", "rv", "bpv")],
        *[pa.field(f"{measure}_{part}", pa.string(), nullable=False)
          for measure in ("log_return", "rv", "bpv") for part in ("status", "reason")],
    ])
    demo_observation_sets = {}
    for window_name, return_curve in [("none", None), *[(window, selected)
        for window in demo_window_names for selected in ("rv", "bpv")]]:
        configuration = "none" if return_curve is None else f"{window_name}_return{return_curve}"
        if configurations is not None and configuration not in configurations:
            continue
        kwargs = {} if window_name == "none" else dict(
            rv_curve=demo_curves[window_name]["rv"], bpv_curve=demo_curves[window_name]["bpv"], return_seasonality=return_curve)
        minute_observations = prepare_minute_observations(demo_contributions, **kwargs)
        for measure, source in (("log_return", "log_return"), ("rv", "rv_contribution"), ("bpv", "bpv_contribution")):
            good = minute_observations[f"{measure}_status"].eq("ok")
            factor_col = "return_seasonal_factor" if measure == "log_return" else f"{measure}_seasonal_factor"
            denominator = minute_observations.loc[good, factor_col].to_numpy()
            if measure == "log_return": denominator = np.sqrt(denominator)
            np.testing.assert_allclose(minute_observations.loc[good, measure],
                demo_contributions.loc[good, source] / denominator)
        observations = aggregate_model_observations(minute_observations, demo_positions)
        assert len(observations) == len(demo_positions)
        np.testing.assert_array_equal(observations["sample_id"], demo_positions["sample_id"])
        # 在完整序列的正覆盖区间逐一核对首尾及中间分钟积分，而非重算采样价格。
        for row in observations.iloc[1:].itertuples():
            left, right = row.interval_start_coordinate, row.interval_end_coordinate
            indices = np.arange(int(np.floor(left)), int(np.ceil(right)))
            weights = np.minimum(indices + 1, right) - np.maximum(indices, left)
            for measure in ("log_return", "rv", "bpv"):
                if getattr(row, f"{measure}_status") == "ok":
                    np.testing.assert_allclose(getattr(row, measure), np.dot(
                        minute_observations.iloc[indices][measure].to_numpy(), weights), rtol=1e-8, atol=1e-12)
        configuration = "none" if return_curve is None else f"{window_name}_return{ return_curve}"
        demo_observation_sets[configuration] = observations
        for measure in ("log_return", "rv", "bpv"):
            transformed = transform_model_input(observations, observation_col=measure,
                require_nonnegative=measure != "log_return")
            np.testing.assert_allclose(transformed["model_input"], observations[measure], equal_nan=True)
        log_inputs = transform_model_input(observations, observation_col="rv", transform="log")
        positive = observations["rv_status"].eq("ok") & observations["rv"].gt(0)
        np.testing.assert_allclose(log_inputs.loc[positive, "model_input"], np.log(observations.loc[positive, "rv"]))
        print(f"{configuration}：{len(observations):,} 点；收益有效 {observations.log_return_status.eq('ok').sum():,}", flush=True)
        del minute_observations
    return dict(observation_sets=demo_observation_sets, positions=demo_positions,
                calibration=demo_calibration, schema=demo_observation_schema, anchor=demo_anchor)


## Demo：既定采样位置上的模型观测

用途：直接消费 c01 的完整分钟贡献、季节曲线和 b03 已验收的采样位置，调用正式函数去季节、按分钟覆盖比例汇总并检查输入尺度转换。输出为不去季节及 1 年／3 年窗口、RV／BPV 收益季节曲线共五组观测。

完整范围为 RB 2010-01-01—2026-09-30，训练截至 2024-12-31，测试自 2025-01-01；使用 N15、GK sigma × 全合约成交额、log_interpolate、一次训练期校准的既定位置，预期 78,007 点。试错截取完整序列的前缀，保留原分钟坐标，不重新采样或校准阈值。完整 DLC 按连续目标区间分片，每片保留一个前置采样位置和完整分钟轴。
上游完整 demo 未就绪时，试错读取全量真实分钟与既定采样位置，再切小样本，调用已有上游函数在内存准备输入。试错不消费来源不匹配的旧成果，不保存结果。正式 DLC 仍须使用已验收的完整上游成果；补齐后重新执行读取输入单元。


## 试错 demo

完整输入只读取一次；以下试错调用正式函数，仅在内存检查和展示。

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
candidate_root = Path.cwd().resolve()
while not (candidate_root / "config/settings.py").is_file():
    if candidate_root == candidate_root.parent:
        raise RuntimeError("找不到项目根目录；请在本项目内打开 Notebook")
    candidate_root = candidate_root.parent
sys.path[:0] = [str(candidate_root)]
from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import (
    _demo_products, _demo_open_run, _demo_upload, _demo_submit, _demo_wait, _demo_recover, _demo_read_saved,
    _read_real_demo, _load_intraday_demo_inputs, _compute_intraday_demo,
    _demo_load_positions, _demo_read_trial_sources, _demo_trial_prefix)
import pyarrow.parquet as pq
from R04_Research.a01_Methods.b03_Sampling.c02_SamplingExecution import read_reusable_sampling_identity
demo_stage = "observations"
full_inputs, trial_sources, full_input_issue = None, None, ""
try:
    seasonality_products = list(_demo_products("seasonality").values())
    contributions_df, contribution_identity = _read_real_demo(candidate_root / seasonality_products[0])
    positions_df, positions_identity = _demo_load_positions(candidate_root)
    full_inputs = dict(frames={"contributions": contributions_df, "positions": positions_df},
        scope=contribution_identity["scope"],
        upstream=dict(contributions=contribution_identity, positions=positions_identity, curves={"1y": {}, "3y": {}}))
    for index, (measure, window) in enumerate([(m, w) for m in ("rv", "bpv") for w in ("1y", "3y")], 1):
        curve_df, curve_identity = _read_real_demo(candidate_root / seasonality_products[index])
        full_inputs["frames"][f"curve_{measure}_{window}"] = curve_df
        full_inputs["upstream"]["curves"][window][measure] = curve_identity
except FileNotFoundError as error:
    full_inputs, full_input_issue = None, str(error)
except RuntimeError as error:
    if not ("计算定义已变化，请重算" in str(error) or "已变化，请重算" in str(error)):
        raise
    full_inputs, full_input_issue = None, str(error)
if full_inputs is None:
    print("c01 完整成果尚未就绪；先用当前函数在内存准备小样本。", full_input_issue)
    trial_sources = _demo_read_trial_sources(candidate_root)
else:
    print(f'完整既定采样位置：{len(full_inputs["frames"]["positions"]):,} 点')


In [ ]:
trial_cases = ("none", "1y_returnrv", "1y_returnbpv", "3y_returnrv", "3y_returnbpv")
if full_inputs is None:
    trial_prefix = _demo_trial_prefix(trial_sources, include_seasonality=True)
    trial_positions_df = trial_prefix["positions"]
    trial_contributions_df = trial_prefix["computed"]["contributions"]
    trial_curve_rows = {f"curve_{m}_{w}": rows for (m, w), rows in trial_prefix["computed"]["curve_diagnostics"].items()}
else:
    trial_positions_df = full_inputs["frames"]["positions"].iloc[:64].copy()
    trial_contributions_df = full_inputs["frames"]["contributions"]
    trial_curve_rows = {key: frame for key, frame in full_inputs["frames"].items() if key.startswith("curve_")}
trial_minute_stop = int(trial_positions_df["source_minute_index"].max()) + 1
trial_contributions_df = trial_contributions_df.iloc[:trial_minute_stop].copy()
# 从坐标 0 起保留分钟前缀，sample_id、source_minute_index 均不重编号。


In [ ]:
trial_tables = {}
for case in trial_cases:
    curve_options = {}
    if case != "none":
        window, selected_curve = case.split("_")[0], case.split("return")[1]
        for measure in ("rv", "bpv"):
            curve_rows = trial_curve_rows[f"curve_{measure}_{window}"].copy()
            curve_rows["trading_date"] = pd.to_datetime(curve_rows["trading_date"])
            curve_options[f"{measure}_curve"] = curve_rows.pivot(index="trading_date",
                columns="minute_slot", values="seasonal_factor")
        curve_options["return_seasonality"] = selected_curve
    trial_minute_df = prepare_minute_observations(trial_contributions_df, **curve_options)
    trial_observations_df = aggregate_model_observations(trial_minute_df, trial_positions_df)
    for row in trial_observations_df.itertuples():
        if not np.isfinite(row.interval_start_coordinate):
            continue
        indices = np.arange(int(np.floor(row.interval_start_coordinate)), int(np.ceil(row.interval_end_coordinate)))
        weights = np.minimum(indices + 1, row.interval_end_coordinate) - np.maximum(indices, row.interval_start_coordinate)
        for measure in ("log_return", "rv", "bpv"):
            if getattr(row, f"{measure}_status") == "ok":
                np.testing.assert_allclose(getattr(row, measure),
                    np.dot(trial_minute_df.iloc[indices][measure].to_numpy(), weights), rtol=1e-8, atol=1e-12)
    for measure in ("log_return", "rv", "bpv"):
        transformed = transform_model_input(trial_observations_df, observation_col=measure,
            require_nonnegative=measure != "log_return")
        np.testing.assert_allclose(transformed.model_input, trial_observations_df[measure], equal_nan=True)
    logged = transform_model_input(trial_observations_df, observation_col="rv", transform="log")
    positive = trial_observations_df.rv_status.eq("ok") & trial_observations_df.rv.gt(0)
    np.testing.assert_allclose(logged.loc[positive, "model_input"], np.log(trial_observations_df.loc[positive, "rv"]))
    trial_tables[case] = trial_observations_df


In [ ]:
demo_display_labels = {
    "sample_id": "sample_id（采样编号）", "sample_at": "sample_at（采样时刻）",
    "trading_date": "trading_date（交易日）", "sample_side": "sample_side（训练／测试侧）",
    "model": "model（模型）", "status": "status（状态）", "reason": "reason（原因）",
    "mean_forecast": "mean_forecast（下一步均值预测）",
    "variance_forecast": "variance_forecast（下一步条件方差）",
    "training_points": "training_points（训练采样点数）",
    "update_status": "update_status（观测更新状态）", "updated_mean_forecast": "updated_mean_forecast（更新后的下一步预测）",
    "parameters": "parameters（具名拟合系数）", "features": "features（HAR 特征）",
    "components": "components（MEM 成分）", "log_return": "log_return（区间对数收益）",
    "rv": "rv（区间平方收益贡献）", "bpv": "bpv（区间双幂变差贡献）",
    "log_return_status": "log_return_status（收益状态）", "rv_status": "rv_status（RV 状态）",
    "bpv_status": "bpv_status（BPV 状态）", "bar_at": "bar_at（分钟末时刻）",
    "rv_contribution": "rv_contribution（分钟平方收益贡献）",
    "bpv_contribution": "bpv_contribution（分钟双幂变差贡献）", "minute_slot": "minute_slot（分钟位置）",
    "seasonal_factor": "seasonal_factor（季节因子）", "causal_factor": "causal_factor（因果季节因子）",
    "is_initial_backfill": "is_initial_backfill（是否初始回填）"}
for configuration, trial_frame in trial_tables.items():
    print(configuration, "；试错结果仅在内存展示")
    selected_columns = [name for name in demo_display_labels if name in trial_frame.columns]
    display(trial_frame.loc[:, selected_columns].iloc[sorted(set(range(min(4, len(trial_frame)))) | set(range(max(0, len(trial_frame) - 4), len(trial_frame))))].rename(columns=demo_display_labels))
    if "status" in trial_frame:
        display(trial_frame.groupby("status", dropna=False).size().rename("行数").rename_axis("status（状态）").to_frame())


## 正式 demo（DLC）

按参数与输入身份复用本地完整成果或接续已有作业；没有才提交。云端只加载导出定义，跳过本地试错、提交、查询及展示。

In [ ]:
demo_parameters = dict(start="2010-01-01", end="2026-09-30", test_start="2025-01-01",
    sampling=dict(N=15, axis="gk_money", quantity_scope="all_contracts", sampling_method="log_interpolate",
                  execution_mode="fixed", K=None),
    cases=["none", "1y_returnrv", "1y_returnbpv", "3y_returnrv", "3y_returnbpv"])
demo_resources = dict(region="cn-shenzhen", workspace_id="351454",
    oss_bucket="oss-pai-q00s96z5hnrhywm9ns-cn-shenzhen",
    image_uri="dsw-registry-vpc.cn-shenzhen.cr.aliyuncs.com/pai/python:3.13.9-cpu-ubuntu22.04-9e1c8c7e-1764329431",
    instance_type="ecs.c6.large", instance_count=10)
demo_recompute = False  # 默认复用／接续；仅明确重算且旧作业已确认终态时改为 True。


In [ ]:
if full_inputs is None:
    raise RuntimeError("完整上游成果尚未就绪。完成上游 DLC 后重新执行读取输入单元；内存试错不会用于完整任务。\n" + full_input_issue)

if demo_stage != "seasonality":
    reference = full_inputs["frames"]["positions" if demo_stage == "observations" else "observations"]
    if len(reference) != 78007:
        raise RuntimeError("本次完整 demo 应覆盖 78,007 个既定采样位置")
demo_context = _demo_open_run(candidate_root, demo_stage, full_inputs, demo_parameters,
                              demo_resources, recompute=demo_recompute)
demo_recompute = False
print("本地完整成果有效，跳过云计算" if demo_context["cached"] else
      f'{demo_context["record"]["name"]} | {demo_context["record"]["state"]}')


In [ ]:
def _observation_demo_chunk(code_root, input_dir, rank, node_count, case):
    """仅分配输出区间；正式分钟轴与阈值不切片、不重新估计。"""
    from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import _demo_node_inputs
    window, selected_curve = ("none", None) if case == "none" else (case.split("_")[0], case.split("return")[1])
    names = ["contributions", "positions"] + ([] if case == "none" else [f"curve_rv_{window}", f"curve_bpv_{window}"])
    plan, frames = _demo_node_inputs(code_root, input_dir, frame_names=names)
    positions = frames["positions"]
    targets = np.array_split(np.arange(len(positions)), node_count)[rank]
    start, stop = (int(targets[0]), int(targets[-1]) + 1) if len(targets) else (0, 0)
    # 多取一个已经确定的前置位置，供正式积分函数确定首个目标区间的起点。
    selected = positions.iloc[max(0, start - 1):stop].copy()
    kwargs = {}
    if selected_curve:
        for measure in ("rv", "bpv"):
            rows = frames[f"curve_{measure}_{window}"].copy()
            rows["trading_date"] = pd.to_datetime(rows["trading_date"])
            kwargs[f"{measure}_curve"] = rows.pivot(index="trading_date", columns="minute_slot", values="seasonal_factor")
        kwargs["return_seasonality"] = selected_curve
    minute = prepare_minute_observations(frames["contributions"], **kwargs)
    result = aggregate_model_observations(minute, selected)
    if start:
        result = result.iloc[1:].copy()
    assert result.sample_id.tolist() == positions.iloc[targets].sample_id.tolist()
    for row in result.itertuples():
        if not np.isfinite(row.interval_start_coordinate):
            continue
        left, right = row.interval_start_coordinate, row.interval_end_coordinate
        indices = np.arange(int(np.floor(left)), int(np.ceil(right)))
        weights = np.minimum(indices + 1, right) - np.maximum(indices, left)
        for measure in ("log_return", "rv", "bpv"):
            if getattr(row, f"{measure}_status") == "ok":
                np.testing.assert_allclose(getattr(row, measure),
                    np.dot(minute.iloc[indices][measure].to_numpy(), weights), rtol=1e-8, atol=1e-12)
        # 这是数学关系验收，不把模型允许保留的失败改成作业失败。
    for measure in ("log_return", "rv", "bpv"):
        transformed = transform_model_input(result, observation_col=measure, require_nonnegative=measure != "log_return")
        np.testing.assert_allclose(transformed.model_input, result[measure], equal_nan=True)
    logged = transform_model_input(result, observation_col="rv", transform="log")
    positive = result.rv_status.eq("ok") & result.rv.gt(0)
    np.testing.assert_allclose(logged.loc[positive, "model_input"], np.log(result.loc[positive, "rv"]))
    schema = pa.schema([*SAMPLING_POSITION_SCHEMA,
        *[pa.field(name, pa.float64()) for name in
          ("interval_start_coordinate", "interval_end_coordinate", "log_return", "rv", "bpv")],
        *[pa.field(f"{measure}_{part}", pa.string(), nullable=False)
          for measure in ("log_return", "rv", "bpv") for part in ("status", "reason")]])
    return case, result, schema


def run_observations_demo_node(code_root, input_dir, output_dir, *, rank, node_count):
    from concurrent.futures import ProcessPoolExecutor
    import multiprocessing
    from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import (
        _demo_node_inputs, _demo_save_node, _demo_stage_node_inputs, _demo_products)
    input_dir = _demo_stage_node_inputs(code_root, input_dir, rank)
    plan, _ = _demo_node_inputs(code_root, input_dir, frame_names=())
    if node_count != 10 or node_count != plan["identity"]["resources"]["instance_count"] or not 0 <= rank < node_count:
        raise ValueError("节点身份不匹配")
    cases = ("none", "1y_returnrv", "1y_returnbpv", "3y_returnrv", "3y_returnbpv")
    products = {}
    with ProcessPoolExecutor(max_workers=2, mp_context=multiprocessing.get_context("spawn")) as pool:
        futures = [pool.submit(_observation_demo_chunk, code_root, input_dir, rank, node_count, case) for case in cases]
        for index, future in enumerate(futures):
            case, frame, schema = future.result()
            window = "none" if case == "none" else case.split("_")[0]
            upstream = plan["metadata"]["upstream"]
            parameters = dict(seasonal_window=window,
                return_seasonality=None if case == "none" else case.split("return")[1],
                sampling=plan["identity"]["parameters"]["sampling"],
                target_partition=dict(rank=rank, node_count=node_count, selection="contiguous",
                    total_target_points=plan["identity"]["frames"]["positions"]["rows"]))
            products[list(_demo_products("observations"))[index]] = (frame, schema, "c02_ModelObservations",
                dict(contributions=upstream["contributions"], positions=upstream["positions"],
                    curves={} if case == "none" else upstream["curves"][window]), parameters, [("sample_id", "ascending")])
    _demo_save_node(code_root, output_dir, plan, rank, products)


In [ ]:
_demo_upload(demo_context)  # 只冻结同一份 full_inputs，不重新读取上游；不上传试错结果。


In [ ]:
demo_job = _demo_submit(demo_context)
print(f'{demo_job["name"]} | {demo_job.get("job_id", "本地成果复用")} | {demo_job.get("console_uri", "")}')
print("输出位置：", demo_job.get("actual_output_uri", demo_job.get("requested_output_uri", "")))


In [ ]:
_demo_wait(demo_context)  # 普通 Python 每 60 秒查同一 ID；明确失败后留证并结束，不重提。


In [ ]:
_demo_recover(demo_context)  # 接收所有节点、核对身份／完整覆盖、安装并复读；失败不重新计算。
print("完整 demo 数据已保存；展示可独立执行最后一格。")


In [ ]:
from pathlib import Path
import sys
from IPython.display import display
candidate_root = Path.cwd().resolve()
while not (candidate_root / "config/settings.py").is_file():
    if candidate_root == candidate_root.parent:
        raise RuntimeError("找不到项目根目录；请在本项目内打开 Notebook")
    candidate_root = candidate_root.parent
sys.path[:0] = [str(candidate_root)]
from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import _demo_read_saved
demo_display_labels = {
    "sample_id": "sample_id（采样编号）", "sample_at": "sample_at（采样时刻）",
    "trading_date": "trading_date（交易日）", "sample_side": "sample_side（训练／测试侧）",
    "model": "model（模型）", "status": "status（状态）", "reason": "reason（原因）",
    "mean_forecast": "mean_forecast（下一步均值预测）",
    "variance_forecast": "variance_forecast（下一步条件方差）",
    "training_points": "training_points（训练采样点数）",
    "update_status": "update_status（观测更新状态）", "updated_mean_forecast": "updated_mean_forecast（更新后的下一步预测）",
    "parameters": "parameters（具名拟合系数）", "features": "features（HAR 特征）",
    "components": "components（MEM 成分）", "log_return": "log_return（区间对数收益）",
    "rv": "rv（区间平方收益贡献）", "bpv": "bpv（区间双幂变差贡献）",
    "log_return_status": "log_return_status（收益状态）", "rv_status": "rv_status（RV 状态）",
    "bpv_status": "bpv_status（BPV 状态）", "bar_at": "bar_at（分钟末时刻）",
    "rv_contribution": "rv_contribution（分钟平方收益贡献）",
    "bpv_contribution": "bpv_contribution（分钟双幂变差贡献）", "minute_slot": "minute_slot（分钟位置）",
    "seasonal_factor": "seasonal_factor（季节因子）", "causal_factor": "causal_factor（因果季节因子）",
    "is_initial_backfill": "is_initial_backfill（是否初始回填）"}
saved_demo_tables = _demo_read_saved(candidate_root, "observations", "c02_ModelObservations")
for filename, saved_frame in saved_demo_tables.items():
    print(filename, f"；完整 {len(saved_frame):,} 行")
    selected_columns = [name for name in demo_display_labels if name in saved_frame.columns]
    display(saved_frame.loc[:, selected_columns].head(8).rename(columns=demo_display_labels))
    if "status" in saved_frame:
        display(saved_frame.groupby("status", dropna=False).size().rename("行数").rename_axis("status（状态）").to_frame())
# 本主题当前提供表格展示；若增加正式绘制函数，在此调用即可，不触发计算或保存试错图。
